In [6]:
import pandas as pd

In [7]:
train_data = pd.read_csv("../datasets/train.csv")
test_data = pd.read_csv("../datasets/test.csv")

In [3]:
train_data.columns

Index(['id', 'age', 'job', 'marital', 'education', 'default', 'balance',
       'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign',
       'pdays', 'previous', 'poutcome', 'y'],
      dtype='object')

In [5]:
test_data.columns

Index(['id', 'age', 'job', 'marital', 'education', 'default', 'balance',
       'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign',
       'pdays', 'previous', 'poutcome'],
      dtype='object')

In [ ]:
for col in train_data.columns:
    print(col)
    print(train_data[col].unique())

In [ ]:
for col in test_data.columns:
    print(col)
    print(test_data[col].unique())

In [10]:
train_data["previous"].unique()

array([  0,   3,   4,   2,   1,   5,   6,  10,  11,   9,   7,  14,  13,
        17,   8,  24,  23,  37,  12,  27,  25,  38,  15,  16,  29,  19,
        20,  32,  55,  18,  34,  22,  26,  21,  28,  35,  39,  51,  31,
        43,  30,  36,  33,  41,  40,  47,  46, 200,  48,  58])

In [12]:
train_data["pdays"].unique()

array([ -1, 175,  91, 181, 252,  90, 211, 183,  87, 168, 172, 364, 266,
       122,  94, 196, 324,  92, 348,  97, 173, 190, 271, 374, 185, 366,
       339, 204, 399, 340,  88, 154,  24, 188, 370, 150, 365, 101, 133,
       355,  83,  99, 247, 259, 350, 189,  95, 180, 367, 371, 265, 151,
       363,  85,  86, 347, 131, 184, 344, 459, 352, 360,  96, 104, 334,
       186, 135, 362,  40, 280, 361, 178, 202, 244, 776, 109, 351, 293,
       182, 192, 353, 356, 187, 369, 191, 170, 255, 317, 174, 321, 169,
       177, 105, 315, 176, 346, 132, 330, 287,  70, 368, 381, 478, 262,
       342, 157, 254, 331, 171, 149, 153,  89, 234, 107,   8,  93, 322,
       253, 103, 245, 251, 127, 336, 357, 258, 226, 197, 140, 250, 354,
       102, 299, 193, 337, 111, 179, 343,   2, 286, 205, 264, 267, 302,
       142, 148, 325,  81, 461, 195, 274, 308, 508, 345, 440, 586, 358,
       222, 332, 314, 200, 263, 272, 305, 289, 298,  98, 238, 294, 160,
       273, 359, 113, 511, 165, 288, 120, 115,   9, 301,   1, 15

In [ ]:
import json
import os
import numpy as np
import pandas as pd


# =========================================================
# DATA CLEANING
# =========================================================

class DataCleaning:
    def __init__(self, file_path, config_path):
        self.file_path = file_path
        self.config_file = config_path
        self.df = None
    
    def load_config(self):
        if not os.path.exists(self.config_file):
            raise FileNotFoundError(f"Config file not found: {self.config_file}")
        
        with open(self.config_file, 'r') as f:
            self.config = json.load(f)
        
        return self.config

    # -----------------------------------------------------
    # Read CSV
    # -----------------------------------------------------

    def read_file(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    # -----------------------------------------------------
    # Clean String Values
    # Example:
    # admin. -> admin
    # blue-collar -> blue_collar
    # -----------------------------------------------------

    def clean_categorical_values(self):
        categorical_cols = self.df.select_dtypes(include='object').columns

        for col in categorical_cols:

            self.df[col] = (
                                self.df[col]
                                .astype(str)
                                .str.strip()
                                .str.lower()
                                .str.replace('.', '', regex=False)
                                .str.replace('-', '_', regex=False)
                            )
        return self.df

    # -----------------------------------------------------
    # Drop Columns
    # -----------------------------------------------------

    def drop_columns(self):
        existing_cols = [
                            col for col in self.config.get("drop_cols", [])
                            if col in self.df.columns
                        ]
        self.df = self.df.drop(columns=existing_cols)
        return self.df

    # -----------------------------------------------------
    # Convert Numeric Columns
    # -----------------------------------------------------

    def convert_numeric_columns(self):
        numeric_columns = self.config.get("numeric_columns", [])
        for col in numeric_columns:
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        return self.df

    # -----------------------------------------------------
    # Complete Cleaning Pipeline
    # -----------------------------------------------------

    def process(self):
        self.read_file()
        self.load_config()
        self.clean_categorical_values()
        self.drop_columns()
        self.convert_numeric_columns()
        return self.df, self.config


# =========================================================
# PREPROCESSING
# =========================================================

class Preprocessing:
    def __init__(self, data, config, output_path='cleaned_data.csv'):
        self.df = data
        self.config = config
        self.output_path = output_path

    # -----------------------------------------------------
    # Binary Encoding
    # yes/no -> 1/0
    # -----------------------------------------------------

    def binary_encoding(self):
        for col in self.config.get("binary_columns"):
            self.df[col] = self.df[col].map({'yes': 1, 'no': 0})
        # return df

    # -----------------------------------------------------
    # Ordinal Encoding
    # -----------------------------------------------------

    def ordinal_encoding(self):
        for col, mapping in self.config.get("ordinal_mappings", {}).items():
            self.df[col] = self.df[col].map(mapping)
        # return self.df

    # -----------------------------------------------------
    # Month Encoding
    # Temporal ordered feature
    # -----------------------------------------------------

    def month_encoding(self):
        self.df['month'] = self.df['month'].map(self.config.get('month_mapping'))
        # return self.df

    # -----------------------------------------------------
    # One Hot Encoding
    # -----------------------------------------------------

    def one_hot_encoding(self):
        self.df = pd.get_dummies(
                                    self.df,
                                    columns=self.config.get('nominal_columns'),
                                    drop_first=False
                                )
        # return df

    # -----------------------------------------------------
    # Feature Engineering
    # -----------------------------------------------------

    def feature_engineering(self):

        # ---------------------------------------------
        # Prior Contacted
        # ---------------------------------------------

        self.df['prior_contacted'] = (self.df['pdays'] != -1).astype(int)

        # ---------------------------------------------
        # Previous Success
        # ---------------------------------------------

        self.df['previous_success_flag'] = (self.df['poutcome_success'] == 1).astype(int)

        # ---------------------------------------------
        # Debt Burden
        # ---------------------------------------------

        self.df['debt_burden'] = (self.df['housing'] + self.df['loan'])

        # ---------------------------------------------
        # Has Any Loan
        # ---------------------------------------------

        self.df['has_any_loan'] = ((self.df['housing'] == 1) | (self.df['loan'] == 1)).astype(int)

        # ---------------------------------------------
        # Balance To Age
        # ---------------------------------------------

        self.df['balance_to_age'] = (self.df['balance'] / (self.df['age'] + 1))

        # ---------------------------------------------
        # Customer Engagement
        # ---------------------------------------------

        self.df['customer_engagement'] = (self.df['duration'] * self.df['previous'])

        # ---------------------------------------------
        # Log Balance
        # ---------------------------------------------

        # self.df['log_balance'] = np.log1p(np.abs(self.df['balance']))

        # return df

    def save_cleaned_data(self):
        self.df.to_csv(self.output_path, index=False)
        print(f"Cleaned data saved to: {self.output_path}")

    # -----------------------------------------------------
    # Complete Preprocessing
    # -----------------------------------------------------

    def process(self):
        self.binary_encoding()
        self.ordinal_encoding()
        self.month_encoding()
        self.one_hot_encoding()
        self.feature_engineering()
        self.save_cleaned_data()

In [13]:
data_cleaner = DataCleaning(
                                file_path="../datasets/train.csv",
                                config_path="../utils/config.json"
                            )

cleaned_data, config_data = data_cleaner.process()

In [14]:
data_preprocessor = Preprocessing(data=cleaned_data, config=config_data, output_path="../datasets/cleaned_train.csv")
data_preprocessor.process()

Cleaned data saved to: ../datasets/cleaned_train.csv


In [13]:
import json
import pandas as pd

In [14]:
X = pd.read_csv("../datasets/cleaned_train.csv")
X = X.drop(columns=['y'])

In [15]:
train_columns = X.columns
with open("../utils/train_columns.json", "w") as f:
    json.dump(list(train_columns), f)